In [1]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
 
import numpy as np
import pandas as pd
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import classification_report, confusion_matrix
import json
 
print(f"TensorFlow {tf.__version__}")


TensorFlow 2.21.0


In [2]:
df = pd.read_csv("medical_test_dataset.csv")
print(f"\nLoaded {len(df)} rows")
print(df["status_label"].value_counts())



Loaded 5000 rows
status_label
normal    3046
high       983
low        971
Name: count, dtype: int64


In [16]:
# Encode test name
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
df["test_name_enc"] = le.fit_transform(df["test_name"])
test_name_classes = le.classes_.tolist()

# Add gaussian noise to value (simulates measurement variability)
noise_std = (df["normal_high"] - df["normal_low"]) * 0.05
df["value"] = df["value"] + np.random.normal(0, noise_std, len(df))
df["value"] = df["value"].clip(lower=0)

# Recalculate derived features AFTER adding noise
df["value_normalized"] = (df["value"] - df["normal_low"]) / \
                         (df["normal_high"] - df["normal_low"])
df["below_normal"] = (df["value"] < df["normal_low"]).astype(int)
df["above_normal"] = (df["value"] > df["normal_high"]).astype(int)
df["deviation_pct"] = (df["value_normalized"] - 0.5) * 2

# Add borderline cases (values within 5% of boundaries — hardest to classify)
borderline_rows = []
for _, row in df.sample(n=500, random_state=42).iterrows():
    low  = row["normal_low"]
    high = row["normal_high"]
    span = high - low

    # Create a borderline sample near a boundary
    side = np.random.choice(["near_low", "near_high"])
    if side == "near_low":
        # Value very close to normal_low — could be Low or Normal
        border_val = low + np.random.uniform(-span*0.05, span*0.05)
        true_status = 0 if border_val < low else 1
    else:
        border_val = high + np.random.uniform(-span*0.05, span*0.05)
        true_status = 2 if border_val > high else 1

    border_val = max(0, border_val)
    v_norm  = (border_val - low) / (high - low)
    new_row = row.copy()
    new_row["value"]           = border_val
    new_row["value_normalized"]= v_norm
    new_row["below_normal"]    = int(border_val < low)
    new_row["above_normal"]    = int(border_val > high)
    new_row["deviation_pct"]   = (v_norm - 0.5) * 2
    new_row["status"]          = true_status
    borderline_rows.append(new_row)

df = pd.concat([df, pd.DataFrame(borderline_rows)], ignore_index=True)
df = df.sample(frac=1, random_state=42).reset_index(drop=True)
print(f"Total samples after augmentation: {len(df)}")
print(df["status"].value_counts())

# ── 3. Features & labels ───────────────────────────────────────────────────────
FEATURES = [
    "test_name_enc",
    "value",
    "normal_low",
    "normal_high",
    "value_normalized",
    "below_normal",
    "above_normal",
    "deviation_pct",
]
X = df[FEATURES].values.astype(np.float32)
y = df["status"].values.astype(np.int32)

# ── 4. Scale ───────────────────────────────────────────────────────────────────
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

with open("scaler_params.json", "w") as f:
    json.dump({
        "mean": scaler.mean_.tolist(),
        "scale": scaler.scale_.tolist(),
        "feature_names": FEATURES
    }, f, indent=2)

with open("test_name_classes.json", "w") as f:
    json.dump(test_name_classes, f)

# ── 5. Split ───────────────────────────────────────────────────────────────────
X_train, X_temp, y_train, y_temp = train_test_split(
    X_scaled, y, test_size=0.3, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)
print(f"\nTrain: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")

# ── 6. Smaller model (reduces overfitting capacity) ───────────────────────────
model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(len(FEATURES),)),

    tf.keras.layers.Dense(32, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.BatchNormalization(),
    tf.keras.layers.Dropout(0.4),          # higher dropout

    tf.keras.layers.Dense(16, activation="relu",
        kernel_regularizer=tf.keras.regularizers.l2(0.001)),
    tf.keras.layers.Dropout(0.3),

    tf.keras.layers.Dense(3, activation="softmax")
], name="MediScanClassifier_v2")

model.summary()

# ── 7. Compile with label smoothing (prevents overconfident predictions) ───────
model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.0005),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(),
    metrics=["accuracy"]
)

# ── 8. Train with proper callbacks ────────────────────────────────────────────
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        patience=15,
        restore_best_weights=True,
        monitor="val_loss",
        min_delta=0.001        # must improve by at least 0.1% to count
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        factor=0.5,
        patience=7,
        monitor="val_loss",
        min_lr=1e-6
    ),
]

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,                # let early stopping decide
    batch_size=128,            # larger batch = less overfitting
    callbacks=callbacks,
    verbose=1
)

# ── 9. Evaluate — key: train vs val gap ───────────────────────────────────────
print("\n── Evaluation ──")
train_loss, train_acc = model.evaluate(X_train, y_train, verbose=0)
val_loss,   val_acc   = model.evaluate(X_val,   y_val,   verbose=0)
test_loss,  test_acc  = model.evaluate(X_test,  y_test,  verbose=0)

print(f"Train accuracy : {train_acc:.4f}")
print(f"Val   accuracy : {val_acc:.4f}")
print(f"Test  accuracy : {test_acc:.4f}")
print(f"Overfit gap    : {train_acc - val_acc:.4f}  (good if < 0.03)")

y_pred = np.argmax(model.predict(X_test, verbose=0), axis=1)
print("\nClassification Report:")
print(classification_report(y_test, y_pred,
      target_names=["Low", "Normal", "High"]))

# ── 10. Convert to TFLite ─────────────────────────────────────────────────────
converter = tf.lite.TFLiteConverter.from_keras_model(model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model = converter.convert()

with open("mediscan_model.tflite", "wb") as f:
    f.write(tflite_model)

print(f"\nModel size: {len(tflite_model)/1024:.1f} KB")
print("✅ Done — copy mediscan_model.tflite, scaler_params.json,")
print("         test_name_classes.json  →  app/src/main/assets/")


Total samples after augmentation: 5500
status
1    3288
2    1112
0    1100
Name: count, dtype: int64

Train: 3850 | Val: 825 | Test: 825


Model: "MediScanClassifier_v2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                         ┃ Output Shape                ┃         Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━┩
│ dense_4 (Dense)                      │ (None, 32)                  │             288 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ batch_normalization_2                │ (None, 32)                  │             128 │
│ (BatchNormalization)                 │                             │                 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_2 (Dropout)                  │ (None, 32)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_5 (Dense)                      │ (None, 16)                  │             528 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dropout_3 (Dropout)                  │ (None, 16)                  │               0 │
├──────────────────────────────────────┼─────────────────────────────┼─────────────────┤
│ dense_6 (Dense)                      │ (None, 3)                   │              51 │
└──────────────────────────────────────┴─────────────────────────────┴─────────────────┘

 Total params: 995 (3.89 KB)

 Trainable params: 931 (3.64 KB)

 Non-trainable params: 64 (256.00 B)

Epoch 1/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 6s 41ms/step - accuracy: 0.5171 - loss: 1.2684 - val_accuracy: 0.6036 - val_loss: 1.0952 - learning_rate: 5.0000e-04
Epoch 2/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.6208 - loss: 1.0134 - val_accuracy: 0.7115 - val_loss: 0.9293 - learning_rate: 5.0000e-04
Epoch 3/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.7003 - loss: 0.8233 - val_accuracy: 0.8388 - val_loss: 0.7553 - learning_rate: 5.0000e-04
Epoch 4/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 15ms/step - accuracy: 0.7691 - loss: 0.6719 - val_accuracy: 0.9176 - val_loss: 0.5995 - learning_rate: 5.0000e-04
Epoch 5/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 17ms/step - accuracy: 0.8164 - loss: 0.5653 - val_accuracy: 0.9588 - val_loss: 0.4743 - learning_rate: 5.0000e-04
Epoch 6/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 0.8395 - loss: 0.5100 - val_accuracy: 0.9661 - val_loss: 0.3805 - learning_rate: 5.0000e-04
Epoch 7/100
31/31 ━━━━━━━━━━━━━━━━━━━━ 1s 16ms/step - accuracy: 

INFO:tensorflow:Assets written to: C:\Users\singh\AppData\Local\Temp\tmpsvjghluy\assets


Saved artifact at 'C:\Users\singh\AppData\Local\Temp\tmpsvjghluy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 8), dtype=tf.float32, name='keras_tensor_9')
Output Type:
  TensorSpec(shape=(None, 3), dtype=tf.float32, name=None)
Captures:
  2775524678864: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775524674832: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775548980560: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775548980944: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775524681936: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775524680016: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775548982480: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775548977872: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775548978256: TensorSpec(shape=(), dtype=tf.resource, name=None)
  2775548977488: TensorSpec(shape=(), dtype=tf.resource, name=None)

Model size: 6.